In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import pyplot
import os
import matplotlib.ticker as ticker
import numpy as np
import xarray as xr
import sys
from utils.utilities import find_best_grid_point, get_station_coords,form_xdate, get_anomalies
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import datetime as dt
import os
import re
import seaborn as sns

from plotting import tol_colors # color schemes from https://personal.sron.nl/~pault/
from utils import process_data
import cmcrameri.cm as cmc


from utils.utilities import get_station_coords

#activate interactive figures
%matplotlib widget
#activate autoreload
%load_ext autoreload

## Add parent directory to syspath
parent_dir = os.path.abspath(os.path.join(os.path.dirname('.'), '..'))
if not parent_dir in sys.path:
    sys.path.append(parent_dir)

dir_data = '/project/leob/GAW/Kenya/data/KMD/tmax'
#dir_data = r'C:\Users\leob\Documents\Data_analyses\Data\KMD\tmax' # local path
#save figures in...
dir_save = './output/kmd_data/'
save_fig = False

In [ ]:
mknlat, mknlon, mknalt = get_station_coords("MKN") 

In [ ]:
precip = xr.open_dataset(f"{dir_data}/Precip/MERU/rr_mrg_20241231_CLM.nc")
plt.figure()
precip.precip.plot(cmap=cmc.oslo_r)
plt.show()

In [ ]:
precip = xr.open_dataset(f"{dir_data}/Precip/ISIOLO/rr_mrg_20241231_CLM.nc")
plt.figure()
precip.precip.plot(cmap=cmc.oslo_r)
plt.show()

In [ ]:
read_original = False # set to True if you want to read from the original files and merge again to new netcdf

save_netcdf = True # if read_from_netcdf is False, save the merged dataset to a netcdf file

# Define the path to the folders and the years of interest
regions = ["ISIOLO","LAIKIPIA", "MERU","NYERI", "SAMBURU"]
vars = ["Precip","tmax", "Tmin"]
# short names are rr for precipitation, tmax for tmax and tmin for Tmin
vars_short = ["rr","tmax", "tmin"]
#folders = [f"{dir_data}/Precip/{r}/" for r in regions]
years = range(2020, 2025)

if read_original:
    # Read all files

    # Initialize an empty list to store the datasets
    datasets_vars = []

    # Loop through each folder and each year to read the netCDF files
    for var, var_short in zip(vars,vars_short):
        datasets_regions = []
        for region in regions:
            datasets = []
            folder = f"{dir_data}/{var}/{region}/"
            for year in years:
                print("Reading data for region:", region, "year:", year)
                # Create a regex pattern for the given year
                pattern = re.compile(f"{var_short}_mrg_{year}[0-9]{{4}}_[A-Z]{{3}}\\.nc$")
                
                # List all netCDF files for the given year in the current folder using regex
                files = [f for f in os.listdir(folder) if pattern.match(f)]
                
                # Loop through each file and open it as an xarray dataset
                for file in files:
                    file_path = os.path.join(folder, file)
                    ds = xr.open_dataset(file_path)
                    
                    # Extract the date from the filename
                    #date = pd.to_datetime(file[7:15]) # extract date from file name
                    match = re.search(r'\d{8}', file)
                    if match:
                        date = pd.to_datetime(match.group(), format='%Y%m%d')
                    else:
                        date = None # or handle the error as needed

                    # Add a time coordinate to the dataset
                    ds = ds.expand_dims({"time": [date]})

                    #rename temp to tmin or tmax
                    if var == "Tmin" or var == "tmax":
                        ds = ds.rename({"temp": var_short})
                    
                    datasets.append(ds)

            # Merge all datasets into a single xarray dataset along the time dimension
            merged_dataset_region = xr.concat(datasets, dim='time')
            # add region dimension
            merged_dataset_region = merged_dataset_region.expand_dims({"region": [region]})
            datasets_regions.append(merged_dataset_region) # merge different regions

        merged_dataset_reg = xr.concat(datasets_regions,dim='region')

        ## Problem: the TMax data has different grid points, so we cannot merge them direcctly
        # Interpolate them to the same grid as precipiation
        if var == 'tmax':
            merged_dataset_reg = merged_dataset_reg.interp(Lon=datasets_vars[0].Lon, Lat=datasets_vars[0].Lat, method="nearest")
        #merged_dataset_vars = merged_dataset_reg.expand_dims({"var": [var]})
        datasets_vars.append(merged_dataset_reg) # merge different regions

    #merged_dataset = xr.concat(datasets_vars,dim='var')
    merged_dataset = xr.merge(datasets_vars) # not working, because Tmax has different grid points!!?

    if save_netcdf:
        merged_dataset.to_netcdf(f"../data/level3/kmd_maproom/MKN_KMD_2020_2024.nc")

else: 
    merged_dataset = xr.open_dataset(f"../data/level3/kmd_maproom/MKN_KMD_2020_2024.nc")


In [ ]:
merged_dataset

In [ ]:
plt.figure()
merged_dataset.isel(time=-1).mean(dim='region').precip.plot(cmap=cmc.oslo_r)
plt.show()

In [ ]:
# check a specific day
for r in regions:
    plt.figure()
    merged_dataset.sel(region=r).isel(time=-1).precip.plot(cmap=cmc.oslo_r)
    plt.plot(
    mknlon, mknlat, marker="o", color="green",zorder=5,markersize=8,
    ) 
    plt.show()

In [ ]:
### Map with mean precipitation
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
import matplotlib.ticker as mticker
from cartopy.feature import ShapelyFeature
from cartopy.io.shapereader import Reader,natural_earth
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import mplotutils as mpu

# area to map
[lon1, lon2, lat1, lat2] = [35, 40, -4, 4] #[33, 43, -6, 6]

# start the figure
projection = ccrs.PlateCarree() # ccrs.Orthographic(central_latitude=40)

fig,ax = plt.subplots(1,1,subplot_kw=dict(projection=projection))
ax.set_extent([lon1, lon2, lat1, lat2], crs=projection)

lands = cfeature.NaturalEarthFeature(
    category="physical", name="land", scale="50m",
)  # cfeature.COLORS['land']

ax.add_feature(cfeature.LAKES)
ax.add_feature(cfeature.RIVERS)

# Add Kenya county borders
filename = "./plotting/kenyan-counties/County.shp"
shape_feature = ShapelyFeature(
    Reader(filename).geometries(),
    ccrs.PlateCarree(),
    facecolor="None",#"whitesmoke",
    edgecolor="dimgrey",
    linestyle="-",
    lw=0.6
)
ax.add_feature(shape_feature)


# # plot town locations
# lons = np.arange(lon1, lon2, 0.1)
# lats = np.arange(lat1, lat2, 0.1)
# plot_towns(ax, lats, lons)

mknlat, mknlon, mknalt = get_station_coords("MKN")
ax.plot(
    mknlon, mknlat, marker="o", color="blue", transform=ccrs.PlateCarree(), zorder=5,markersize=8,
)  # mt. kenya station

### plot precipitation data (e.g. a specific day)
# for r in regions:
#     h = merged_dataset.sel(region=r).isel(time=-1).precip.plot(ax=ax, cmap=cmc.oslo_r,vmin=0, vmax=10, add_colorbar=False, transform=ccrs.PlateCarree())
h = merged_dataset.mean(dim='region').mean(dim='time')['precip'].plot(ax=ax, cmap=cmc.oslo_r,add_colorbar=False, transform=ccrs.PlateCarree())
### Ticks and labels
#ax.set_yticks(np.arange(round(lat1, 1), round(lat2, 1), 2), crs=projection)
#ax.set_yticks([0], crs=projection) # only at 0deg
#ax.set_xticks(np.arange(round(lon1), round(lon2), 2), crs=projection)
# format the ticks
ax.yaxis.set_major_formatter(LatitudeFormatter())
#ax.xaxis.set_major_formatter(LongitudeFormatter())

##grid lines:
gl = ax.gridlines(linewidth=.9, linestyle=':',draw_labels=True,color='grey')
#gl.xlocator = mticker.FixedLocator(np.arange(-180,181,30))
gl.bottom_labels = False
gl.top_labels = False
gl.right_labels = False
gl.left_labels = False # set true if I want the equator tick
gl.xlines = False
gl.ylocator = mticker.FixedLocator([0])
gl.xformatter = LONGITUDE_FORMATTER
gl.yformatter = LATITUDE_FORMATTER

# fig.subplots_adjust(
#    left=0.1, right=0.11, bottom=0.05, top=0.1
# )  # manually adjust spaces to have no cutoff when saving
mpu.set_map_layout(ax, width=12)


ax.spines[:].set_visible(False) # no border


plt.colorbar(h, orientation="vertical", label="(mm)")

In [ ]:
## map with mean Tmin

from matplotlib.colors import TwoSlopeNorm

# area to map
[lon1, lon2, lat1, lat2] = [35, 40, -4, 4]  # [33, 43, -6, 6]

# start the figure
projection = ccrs.PlateCarree()  # ccrs.Orthographic(central_latitude=40)

# Define what to plot in subplots: 
## One variable but different periods
time_start=['2020-01-01', '2022-01-01', '2023-04-01']
time_end=['2021-12-31', '2023-03-31','2024-12-31']
vars = ['tmin','tmin','tmin']
common_colorbar = True

# several variables but one period
time_start=['2020-01-01','2020-01-01']
time_end=['2024-12-31','2024-12-31']
vars = ['tmin','tmax']
common_colorbar = False

fig, axs = plt.subplots(1, len(vars), subplot_kw=dict(projection=projection))
for ax,var, t1,t2 in zip(axs, vars, time_start, time_end):
    ax.set_extent([lon1, lon2, lat1, lat2], crs=projection)

    lands = cfeature.NaturalEarthFeature(
        category="physical",
        name="land",
        scale="50m",
    )  # cfeature.COLORS['land']

    ax.add_feature(cfeature.LAKES)
    ax.add_feature(cfeature.RIVERS)

    # Add Kenya county borders
    filename = "./plotting/kenyan-counties/County.shp"
    shape_feature = ShapelyFeature(
        Reader(filename).geometries(),
        ccrs.PlateCarree(),
        facecolor="None",  # "whitesmoke",
        edgecolor="dimgrey",
        linestyle="-",
        lw=0.6,
    )
    ax.add_feature(shape_feature)


    # # plot town locations
    # lons = np.arange(lon1, lon2, 0.1)
    # lats = np.arange(lat1, lat2, 0.1)
    # plot_towns(ax, lats, lons)

    mknlat, mknlon, mknalt = get_station_coords("MKN")
    ax.plot(
        mknlon,
        mknlat,
        marker="o",
        color="blue",
        transform=ccrs.PlateCarree(),
        zorder=5,
        markersize=8,
    )  # mt. kenya station

    ### plot precipitation data (e.g. a specific day)
    # for r in regions:
    #     h = merged_dataset.sel(region=r).isel(time=-1).precip.plot(ax=ax, cmap=cmc.oslo_r,vmin=0, vmax=10, add_colorbar=False, transform=ccrs.PlateCarree())
    h = (
        merged_dataset.sel(time=slice(t1,t2)).mean(dim="region")
        .mean(dim="time")[var]
        .plot(
            ax=ax,
            cmap=cmc.bilbao_r,
            add_colorbar=False,
            transform=ccrs.PlateCarree(),
            #norm=TwoSlopeNorm(0),
            vmin=0, 
            vmax=40,
            rasterized=True
        )
    )
    ax.set_title(f"{var}, {t1} to {t2}")
    ### Ticks and labels
    # ax.set_yticks(np.arange(round(lat1, 1), round(lat2, 1), 2), crs=projection)
    # ax.set_yticks([0], crs=projection) # only at 0deg
    # ax.set_xticks(np.arange(round(lon1), round(lon2), 2), crs=projection)
    # format the ticks
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    # ax.xaxis.set_major_formatter(LongitudeFormatter())

    ##grid lines:
    gl = ax.gridlines(linewidth=0.9, linestyle=":", draw_labels=True, color="grey")
    # gl.xlocator = mticker.FixedLocator(np.arange(-180,181,30))
    gl.bottom_labels = False
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = False  # set true if I want the equator tick
    gl.xlines = False
    gl.ylocator = mticker.FixedLocator([0])
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER

    # fig.subplots_adjust(
    #    left=0.1, right=0.11, bottom=0.05, top=0.1
    # )  # manually adjust spaces to have no cutoff when saving

    ax.spines[:].set_visible(False)  # no border

    if common_colorbar==False: 
        cbar = mpu.colorbar(h, ax, orientation="vertical", aspect=40, label="(°C)")

    mpu.set_map_layout(ax, width=30)

if common_colorbar:
    #plt.colorbar(h, orientation="vertical", label="(°C)")
    cbar = mpu.colorbar(h, ax, orientation="vertical", aspect=40,label="(°C)")

if save_fig: 
    
    plt.savefig(f"{dir_save}/MKN_KMD_map_{'_'.join(vars)}_{time_start[0]}_{time_end[-1]}.pdf", dpi=300)
plt.show()

In [ ]:
# Mean precipitation
plt.figure()
merged_dataset.mean(dim='region').mean(dim='time').precip.plot(cmap=cmc.oslo_r)
plt.show()

In [ ]:
## monthly precipiation sums
# sum up the precipitation for each month
precip_m = merged_dataset['precip'].mean(dim='region').resample(time='M').sum(dim='time')

In [ ]:
# make a barplot with monthly mean precipitation
import seaborn as sns
# mean of monthly precipitation somes over all years
precip_m_grouped = precip_m.groupby('time.month').mean(dim='time').mean(dim='Lon').mean(dim='Lat') # problem: this takes the sum over months of all years!!

plt.figure()
sns.barplot(data = precip_m_grouped.to_dataframe(),x='month',y='precip')
plt.show()

In [ ]:
# same but only for MKN grid
precip_m_grouped = precip_m.sel(Lat= mknlat, Lon=mknlon,method='nearest').groupby('time.month').mean(dim='time')

plt.figure()
sns.barplot(data = precip_m_grouped.to_dataframe(),x='month',y='precip')
plt.title(f"Gridded monthly precipitation (2020-2024) \n at MKN grid point ({precip_m_grouped.Lat.values:.2f}° lat, {precip_m_grouped.Lon.values:.2f})° lon")
plt.ylabel("Merged station-satellite rainfall (mm)")
plt.show()

In [ ]:
# select the 9 gric cells around MKN
dxy = 0.18 # grid size
plt.figure()
precip_mkn = precip_m.sel(Lat=slice(-0.3,0.2), Lon=slice(37,37.5))
precip_mkn.isel(time=-1).plot(cmap=cmc.oslo_r) # same as Lat=slice(mknlat-dxy,mknlat+dxy*2), Lon=slice(mknlon-dxy*2, mknlon+dxy
plt.plot(
    mknlon, mknlat, marker="o", color="green",  zorder=5,markersize=8,
) 
plt.show()

In [ ]:
# same but only for 9 cells around MKN grid

precip_m_grouped = precip_m.sel(Lat=slice(-0.3,0.2), Lon=slice(37,37.5)).groupby('time.month').mean(dim='time').mean(dim='Lon').mean(dim='Lat')

plt.figure()
sns.barplot(data = precip_m_grouped.to_dataframe(),x='month',y='precip')
plt.title(f"Gridded monthly precipitation (2020-2024) \n at MKN surrounding 9 grid points")
plt.ylabel("Merged station-satellite rainfall (mm)")
plt.show()

In [ ]:
#Monthly precipitaiton timeseries for MKN
precip_m_ts = precip_m.sel(Lat=slice(-0.3,0.2), Lon=slice(37,37.5)).mean(dim='Lat').mean(dim='Lon')
plt.figure()
sns.barplot(data = precip_m_ts.to_dataframe(),x='time',y='precip')
plt.show

In [ ]:
## Seasonal temperature cycle
# same but only for MKN grid
tmin_m_grouped = merged_dataset['tmin'].mean(dim='region').sel(Lat= mknlat, Lon=mknlon,method='nearest').groupby('time.month').mean(dim='time')
plt.figure()
tmin_m_grouped.plot()
plt.show()


In [ ]:
## Initial figure for paper (strange jump in Tmin!?)
import matplotlib.dates as mdates
from plotting import tol_colors # color schemes from https://sronpersonalpages.nl/~pault

col_bright = tol_colors.tol_cset('bright')
## 1 Figure with all variables, time series and seasonal cycle

# for precipitation, choose 9 grids around MKN
mkn_9grids_lats = slice(-0.3,0.2)
mkn_9grids_lons = slice(37,37.25)

mkn_closest_lat = merged_dataset.Lat.values[3] #-0.035
mkn_closest_lon = merged_dataset.Lon.values[6] #37.245002

# select MKN grids 
## monthly precipiation sums, sum up the precipitation for each month
precip_m_sel = merged_dataset['precip'].sel(region='MERU').resample(time='MS').sum(dim='time').sel(Lat=mkn_9grids_lats, Lon=mkn_9grids_lons)
precip_sel = merged_dataset['precip'].sel(region='MERU').sel(Lat=mkn_9grids_lats, Lon=mkn_9grids_lons)

ds_sel = merged_dataset.sel(region='MERU').sel(Lat= mkn_closest_lat, Lon=mkn_closest_lon)

# Time series
fig, axs = plt.subplots(2, 1,)

# Time series
ax1 = axs[0]
ds_sel['tmax'].plot(ax=ax1, color=col_bright.red, label='Tmax')
ds_sel['tmin'].plot(ax=ax1, color=col_bright.cyan, label ='Tmin')
ax1.set_title('')
ax1.legend(loc='upper left', fontsize=8)
ax1.set_xlabel('')

ax12 = ax1.twinx()
precip_color = col_bright.blue #(0, 0, 1, 0.6)
precip_sel.mean(dim='Lat').mean(dim='Lon').plot(ax=ax12, label='Precipitation',alpha=0.6, color=col_bright.blue)
ax12.set_title('')
ax12.set_ylabel('Daily precipitation (mm)')
ax12.set_ylim(0, 200)
ax12.legend(ncols=2, loc='upper right', fontsize=8)
#plt.bar(precip_m_sel.time.values,precip_m_sel.mean(dim='Lat').mean(dim='Lon').values,ax=ax2)
#sns.barplot(data = precip_m_sel.mean(dim='Lat').mean(dim='Lon').to_dataframe(),x='time',y='precip')
# color right axis
ax12.spines['right'].set_color(precip_color)
ax12.yaxis.label.set_color(precip_color)
ax12.tick_params(axis='y', colors=precip_color)


ax12.xaxis.set_major_locator(mdates.YearLocator(1))
ax12.xaxis.set_minor_locator(mdates.MonthLocator())
ax12.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))


## seasonal cycles /monthly data
months = range(1, 13)

precip_m_grouped = precip_m_sel.groupby('time.month').mean(dim='time').mean(dim='Lon').mean(dim='Lat')
tmin_m_grouped = ds_sel['tmin'].groupby('time.month').mean(dim='time')
tmax_m_grouped = ds_sel['tmax'].groupby('time.month').mean(dim='time')

ax2 = axs[1]
ax2.plot(months, tmin_m_grouped, color=col_bright.cyan, label='Tmin')
ax2.plot(months, tmax_m_grouped, color=col_bright.red, label='Tmax')
#sns.lineplot(data = ds_sel.groupby('time.month').mean(dim='time')['tmin'].to_dataframe(),x='month',y='tmin', ax=axs[1], color='red')
ax2.set_title('')
ax2.set_ylim(0,25)
ax2.legend(loc='upper left', fontsize=8)
ax2.set_ylabel('Temperature (°C)')
ax2.set_xlabel('Month')

ax22 = ax2.twinx()
ax22.bar(months, precip_m_grouped, color=col_bright.blue, alpha=0.6, label='Precipitation')
#sns.barplot(data = precip_m_grouped.to_dataframe(),x='month',y='precip', ax=ax22)
ax22.set_title('')
ax22.set_ylim(0, 300)
ax22.legend(ncols=2, loc='upper right', fontsize=8)
# color right axis
ax22.spines['right'].set_color(precip_color)
ax22.yaxis.label.set_color(precip_color)
ax22.tick_params(axis='y', colors=precip_color)
ax22.set_ylabel('Monthly precipitation (mm)',color=precip_color)

ax22.set_xticks(months,
                        [dt.datetime.strptime(str(month), "%m").strftime("%b") for month in months],
                    )

plt.suptitle(f"Gridded temperature and precipitation (2020-2024) \n at Mt. Kenya")
dir_save = 'output/publication_figures/' 
if save_fig: 
    plt.savefig(f"{dir_save}/MKN_KMD_2020_2024.pdf", dpi=300)

In [ ]:
# updated figure
# just choose the nearest MKN grid
import matplotlib.dates as mdates
from plotting import tol_colors # color schemes from https://sronpersonalpages.nl/~pault

col_bright = tol_colors.tol_cset('bright')
## 1 Figure with all variables, time series and seasonal cycle

# for precipitation, choose 9 grids around MKN
mkn_9grids_lats = slice(-0.3,0.2)
mkn_9grids_lons = slice(37,37.25)

mkn_closest_lat = merged_dataset.Lat.values[3] #-0.035
mkn_closest_lon = merged_dataset.Lon.values[6] #37.245002

# select MKN grids 
## monthly precipiation sums, sum up the precipitation for each month
# take mean of all regions (there should be no overlap)
precip_m_sel = merged_dataset['precip'].mean(dim='region').resample(time='MS').sum(dim='time').sel(Lat=mknlat, Lon=mknlon, method='nearest')
precip_sel = merged_dataset['precip'].mean(dim='region').sel(Lat=mknlat, Lon=mknlon, method='nearest')

ds_sel = merged_dataset.mean(dim='region').sel(Lat=mknlat, Lon=mknlon,method='nearest')

# select MERU region only
precip_m_sel = merged_dataset['precip'].sel(region='MERU').resample(time='MS').sum(dim='time').sel(Lat=mknlat, Lon=mknlon, method='nearest')
precip_sel = merged_dataset['precip'].sel(region='MERU').sel(Lat=mknlat, Lon=mknlon, method='nearest')
ds_sel = merged_dataset.sel(region='MERU').sel(Lat=mknlat, Lon=mknlon,method='nearest')


# Time series
fig, axs = plt.subplots(2, 1,)

# Time series
ax1 = axs[0]
ds_sel['tmax'].plot(ax=ax1, color=col_bright.red, label='Tmax')
ds_sel['tmin'].plot(ax=ax1, color=col_bright.cyan, label ='Tmin')
ax1.set_title('')
ax1.legend(loc='upper left', fontsize=8)
ax1.set_xlabel('')

ax12 = ax1.twinx()
precip_color = col_bright.blue #(0, 0, 1, 0.6)
precip_sel.plot(ax=ax12, label='Precipitation',alpha=0.6, color=col_bright.blue) #.mean(dim='Lat').mean(dim='Lon')
ax12.set_title('')
ax12.set_ylabel('Daily precipitation (mm)')
ax12.set_ylim(0, 200)
ax12.legend(ncols=2, loc='upper right', fontsize=8)
#plt.bar(precip_m_sel.time.values,precip_m_sel.mean(dim='Lat').mean(dim='Lon').values,ax=ax2)
#sns.barplot(data = precip_m_sel.mean(dim='Lat').mean(dim='Lon').to_dataframe(),x='time',y='precip')
# color right axis
ax12.spines['right'].set_color(precip_color)
ax12.yaxis.label.set_color(precip_color)
ax12.tick_params(axis='y', colors=precip_color)


ax12.xaxis.set_major_locator(mdates.YearLocator(1))
ax12.xaxis.set_minor_locator(mdates.MonthLocator())
ax12.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))


## seasonal cycles /monthly data
months = range(1, 13)

precip_m_grouped = precip_m_sel.groupby('time.month').mean(dim='time')#.mean(dim='Lon').mean(dim='Lat')
tmin_m_grouped = ds_sel['tmin'].groupby('time.month').mean(dim='time')
tmax_m_grouped = ds_sel['tmax'].groupby('time.month').mean(dim='time')

ax2 = axs[1]
ax2.plot(months, tmin_m_grouped, color=col_bright.cyan, label='Tmin')
ax2.plot(months, tmax_m_grouped, color=col_bright.red, label='Tmax')
#sns.lineplot(data = ds_sel.groupby('time.month').mean(dim='time')['tmin'].to_dataframe(),x='month',y='tmin', ax=axs[1], color='red')
ax2.set_title('')
ax2.set_ylim(0,25)
ax2.legend(loc='upper left', fontsize=8)
ax2.set_ylabel('Temperature (°C)')
ax2.set_xlabel('Month')

ax22 = ax2.twinx()
ax22.bar(months, precip_m_grouped, color=col_bright.blue, alpha=0.6, label='Precipitation')
#sns.barplot(data = precip_m_grouped.to_dataframe(),x='month',y='precip', ax=ax22)
ax22.set_title('')
ax22.set_ylim(0, 300)
ax22.legend(ncols=2, loc='upper right', fontsize=8)
# color right axis
ax22.spines['right'].set_color(precip_color)
ax22.yaxis.label.set_color(precip_color)
ax22.tick_params(axis='y', colors=precip_color)
ax22.set_ylabel('Monthly precipitation (mm)',color=precip_color)

ax22.set_xticks(months,
                        [dt.datetime.strptime(str(month), "%m").strftime("%b") for month in months],
                    )

plt.suptitle(f"Gridded temperature and precipitation (2020-2024) \n at Mt. Kenya")
dir_save = 'output/publication_figures/' 
if save_fig: 
    plt.savefig(f"{dir_save}/MKN_KMD_2020_2024.pdf", dpi=300)

In [ ]:
# Same but with meteo data in addition
# just choose the nearest MKN grid
from input import get_meteo
data_path = "../data/"
ds_meteo_new = get_meteo.get_meteo_timeseries_new(data_path=data_path) #starting in 2023
ds_meteo_old = get_meteo.get_meteo_timeseries_old(data_path=data_path) #until 2022
# attention, the data is not processed! seem to have some duplicates in dates?

# remove date duplicates
#ds_meteo_new = ds_meteo_new.drop_duplicates('time')
#ds_meteo_old = ds_meteo_old.drop_duplicates('time')
# merge the two datasets
ds_meteo = xr.concat([ds_meteo_old, ds_meteo_new], dim='time')
# restrict meteo data to 2022 to 2024 (no data before?)
meteo_t1 = '2022-01-01'
meteo_t2 = '2024-12-31'


col_bright = tol_colors.tol_cset('bright')
## 1 Figure with all variables, time series and seasonal cycle

# for precipitation, choose 9 grids around MKN
mkn_9grids_lats = slice(-0.3,0.2)
mkn_9grids_lons = slice(37,37.25)

mkn_closest_lat = merged_dataset.Lat.values[3] #-0.035
mkn_closest_lon = merged_dataset.Lon.values[6] #37.245002

# select MKN grids 
## monthly precipiation sums, sum up the precipitation for each month
# take mean of all regions (there should be no overlap)
precip_m_sel = merged_dataset['precip'].mean(dim='region').resample(time='MS').sum(dim='time').sel(Lat=mknlat, Lon=mknlon, method='nearest')
precip_sel = merged_dataset['precip'].mean(dim='region').sel(Lat=mknlat, Lon=mknlon, method='nearest')

ds_sel = merged_dataset.mean(dim='region').sel(Lat=mknlat, Lon=mknlon,method='nearest')

# select MERU region only
precip_m_sel = merged_dataset['precip'].sel(region='MERU').resample(time='MS').sum(dim='time').sel(Lat=mknlat, Lon=mknlon, method='nearest')
precip_sel = merged_dataset['precip'].sel(region='MERU').sel(Lat=mknlat, Lon=mknlon, method='nearest')
ds_sel = merged_dataset.sel(region='MERU').sel(Lat=mknlat, Lon=mknlon,method='nearest')


# Time series
fig, axs = plt.subplots(2, 1,)

# Time series
ax1 = axs[0]
ds_sel['tmax'].plot(ax=ax1, color=col_bright.red, label='Tmax')
ds_sel['tmin'].plot(ax=ax1, color=col_bright.cyan, label ='Tmin')

# meteo data
ds_meteo['temperature'].sel(time=slice(meteo_t1,meteo_t2)).plot(ax=ax1, label='meteo station MKN',c='k',alpha=0.8,lw=0.8) #.resample(time='1D').mean()

ax1.set_title('')
ax1.legend(loc='upper left', fontsize=8)
ax1.set_xlabel('')
ax1.set_ylabel('Daily temperature (°C)')


ax12 = ax1.twinx()
precip_color = col_bright.blue #(0, 0, 1, 0.6)
precip_sel.plot(ax=ax12, label='Precipitation',alpha=0.6, color=col_bright.blue) #.mean(dim='Lat').mean(dim='Lon')
ax12.set_title('')
ax12.set_ylabel('Daily precipitation (mm)')
ax12.set_ylim(0, 200)
ax12.legend(ncols=2, loc='upper right', fontsize=8)
#plt.bar(precip_m_sel.time.values,precip_m_sel.mean(dim='Lat').mean(dim='Lon').values,ax=ax2)
#sns.barplot(data = precip_m_sel.mean(dim='Lat').mean(dim='Lon').to_dataframe(),x='time',y='precip')
# color right axis
ax12.spines['right'].set_color(precip_color)
ax12.yaxis.label.set_color(precip_color)
ax12.tick_params(axis='y', colors=precip_color)


ax12.xaxis.set_major_locator(mdates.YearLocator(1))
ax12.xaxis.set_minor_locator(mdates.MonthLocator())
ax12.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))


## seasonal cycles /monthly data
months = range(1, 13)

precip_m_grouped = precip_m_sel.groupby('time.month').mean(dim='time')#.mean(dim='Lon').mean(dim='Lat')
tmin_m_grouped = ds_sel['tmin'].groupby('time.month').mean(dim='time')
tmax_m_grouped = ds_sel['tmax'].groupby('time.month').mean(dim='time')

t_meteo_grouped = ds_meteo['temperature'].sel(time=slice(meteo_t1,meteo_t2)).groupby('time.month').mean(dim='time')

ax2 = axs[1]
ax2.plot(months, tmin_m_grouped, color=col_bright.cyan, label='Tmin')
ax2.plot(months, tmax_m_grouped, color=col_bright.red, label='Tmax')
ax2.plot(months, t_meteo_grouped, color='k', label='T meteo')
#sns.lineplot(data = ds_sel.groupby('time.month').mean(dim='time')['tmin'].to_dataframe(),x='month',y='tmin', ax=axs[1], color='red')
ax2.set_title('')
ax2.set_ylim(0,25)
ax2.legend(loc='upper left', fontsize=8)
ax2.set_ylabel('Temperature (°C)')
ax2.set_xlabel('Month')

ax22 = ax2.twinx()
ax22.bar(months, precip_m_grouped, color=col_bright.blue, alpha=0.6, label='Precipitation')
#sns.barplot(data = precip_m_grouped.to_dataframe(),x='month',y='precip', ax=ax22)
ax22.set_title('')
ax22.set_ylim(0, 300)
ax22.legend(ncols=2, loc='upper right', fontsize=8)
# color right axis
ax22.spines['right'].set_color(precip_color)
ax22.yaxis.label.set_color(precip_color)
ax22.tick_params(axis='y', colors=precip_color)
ax22.set_ylabel('Monthly precipitation (mm)',color=precip_color)

ax22.set_xticks(months,
                        [dt.datetime.strptime(str(month), "%m").strftime("%b") for month in months],
                    )

plt.suptitle(f"Gridded temperature and precipitation (2020-2024) \n at Mt. Kenya")
dir_save = 'output/publication_figures/' 
if save_fig: 
    plt.savefig(f"{dir_save}/MKN_KMD_2020_2024_with_meteo.pdf", dpi=300)

In [ ]:
plt.figure()
merged_dataset.sel(region='MERU').sel(Lat= mkn_closest_lat, Lon=mkn_closest_lon)['tmin'].plot()
plt.show()

In [ ]:
merged_dataset.sel(region='MERU').sel(Lat= mkn_closest_lat, Lon=mkn_closest_lon)['tmin'].min()

In [ ]:
# check Tmin
test = merged_dataset.sel(region='MERU') #.sel(Lat= mknlat, Lon=mknlon,method='nearest')

plt.figure()
test.isel(time=-100)['tmin'].plot(cmap=cmc.oslo_r) # same as Lat=slice(mknlat-dxy,mknlat+dxy*2), Lon=slice(mknlon-dxy*2, mknlon+dxy
plt.plot(
    mknlon, mknlat, marker="o", color="green",  zorder=5,markersize=8,
) 
plt.show()


In [ ]:
# check Tmin
test = merged_dataset.mean(dim='region') #.sel(Lat= mknlat, Lon=mknlon,method='nearest')

plt.figure()
test.isel(time=-100)['tmin'].plot(cmap=cmc.oslo_r) # same as Lat=slice(mknlat-dxy,mknlat+dxy*2), Lon=slice(mknlon-dxy*2, mknlon+dxy
plt.plot(
    mknlon, mknlat, marker="o", color="green",  zorder=5,markersize=8,
) 
plt.show()


In [ ]:
plt.figure()
merged_dataset.sel(region='MERU').mean(dim='Lat').mean(dim='Lon').tmin.plot()
plt.show()

In [ ]:
test

In [ ]:

## Plot a map of Kenya

from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import matplotlib.ticker as mticker
from cartopy.feature import ShapelyFeature
from cartopy.io.shapereader import Reader,natural_earth
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
import mplotutils as mpu

# area to map
[lon1, lon2, lat1, lat2] = [33, 43, -6, 6]

# start the figure
projection = ccrs.PlateCarree() # ccrs.Orthographic(central_latitude=40)

fig,ax = plt.subplots(1,1,subplot_kw=dict(projection=projection))
ax.set_extent([lon1, lon2, lat1, lat2], crs=projection)

lands = cfeature.NaturalEarthFeature(
    category="physical", name="land", scale="50m",
)  # cfeature.COLORS['land']
#ax.add_feature(lands, zorder=0, facecolor="white",edgecolor="dimgrey",lw=lw_thin)
#ax.add_feature(cfeature.BORDERS, linestyle="-", edgecolor="dimgrey",lw=lw_thin)

#waters = cfeature.NaturalEarthFeature(category='physical', name='rivers_lake_centerlines',scale= '50m')
#ax.add_feature(waters)

# ax.add_feature(cfeature.LAKES)
# ax.add_feature(cfeature.RIVERS)

# Add Kenya county borders
filename = "./plotting/kenyan-counties/County.shp"
shape_feature = ShapelyFeature(
    Reader(filename).geometries(),
    ccrs.PlateCarree(),
    facecolor="none",#"whitesmoke",
    edgecolor="dimgrey",
    linestyle="-",
    lw=0.6
)
ax.add_feature(shape_feature)

# plot precipitation data
p = merged_dataset.mean(dim='region').mean(dim='time').precip.plot(ax=ax,cmap=cmc.oslo_r,zorder=4, add_colorbar=False)


# # plot town locations
# lons = np.arange(lon1, lon2, 0.1)
# lats = np.arange(lat1, lat2, 0.1)
# plot_towns(ax, lats, lons)

mknlat, mknlon, mknalt = get_station_coords("MKN")
ax.plot(
    mknlon, mknlat, marker="o", color="green", transform=ccrs.PlateCarree(), zorder=5,markersize=8,
)  # mt. kenya station

### Ticks and labels
#ax.set_yticks(np.arange(round(lat1, 1), round(lat2, 1), 2), crs=projection)
#ax.set_yticks([0], crs=projection) # only at 0deg
#ax.set_xticks(np.arange(round(lon1), round(lon2), 2), crs=projection)
# format the ticks
ax.yaxis.set_major_formatter(LatitudeFormatter())
#ax.xaxis.set_major_formatter(LongitudeFormatter())

##grid lines:
gl = ax.gridlines(linewidth=.9, linestyle=':',draw_labels=True,color='grey')
#gl.xlocator = mticker.FixedLocator(np.arange(-180,181,30))
gl.bottom_labels = False
gl.top_labels = False
gl.right_labels = False
gl.left_labels = False # set true if I want the equator tick
gl.xlines = False
gl.ylocator = mticker.FixedLocator([0])
gl.xformatter = LONGITUDE_FORMATTER
gl.yformatter = LATITUDE_FORMATTER

# fig.subplots_adjust(
#    left=0.1, right=0.11, bottom=0.05, top=0.1
# )  # manually adjust spaces to have no cutoff when saving
mpu.set_map_layout(ax, width=15)
mpu.colorbar(p, ax, orientation="vertical", shrink=0.5, shift="symmetric")

ax.spines[:].set_visible(False) # no border

ax.set_title("Average rainfall (2020-2024) \n (Isiolo, Laikipia, Meru, Nyeri, Samburu)", fontsize=14, loc="center")

# if save_fig:
#     #plt.savefig(f"{dir_save}map_kenya_with_lakes_and_stations.{fig_format}",transparent=True, dpi=fig_dpi+150)
#     plt.savefig(f"{dir_save}map_kenya_with_lakes_and_MKNstation.{fig_format}",transparent=True, dpi=fig_dpi+150)
plt.show()